In [17]:
import time
import numpy as np
from brainflow.board_shim import BoardShim, BrainFlowInputParams, BoardIds
from brainflow.data_filter import (
    DataFilter,
    FilterTypes,
    DetrendOperations,
    WindowOperations,
)


def main():
    # ---- 1. 初始化 ----
    BoardShim.enable_dev_board_logger()

    # 选择板卡（这里用模拟板做演示，无需真实硬件）
    board_id = BoardIds.SYNTHETIC_BOARD.value

    # 串口参数（真实板卡需要配置）
    params = BrainFlowInputParams()
    params.serial_port = "COM5"  # Windows 用 "COM3" 等

    # ---- 2. 打印板卡信息 ----
    print("=" * 50)
    # print("板卡描述:", BoardShim.get_board_descr(board_id))
    print("采样率:", BoardShim.get_sampling_rate(board_id), "Hz")
    print("EEG 通道:", BoardShim.get_eeg_channels(board_id))
    print("EXG 通道:", BoardShim.get_exg_channels(board_id))
    print("加速度通道:", BoardShim.get_accel_channels(board_id))
    print("时间戳通道:", BoardShim.get_timestamp_channel(board_id))
    print("标记通道:", BoardShim.get_marker_channel(board_id))
    print("总行数:", BoardShim.get_num_rows(board_id))
    print("加速度通道:", BoardShim.get_accel_channels(board_id))
    print("=" * 50)

    # ---- 3. 创建实例并开始采集 ----
    board = BoardShim(board_id, params)

    try:
        board.prepare_session()
        print("会话已准备 ✓")

        # 开始流式采集，缓冲区 45000 个采样点
        board.start_stream(45000)
        print("数据流已启动，采集 5 秒...\n")

        # 等待数据积累
        time.sleep(5)

        # ---- 4. 获取数据 ----
        data = board.get_current_board_data(256)  # 取最近 256 个采样点
        print(f"数据形状: {data.shape}  (行={data.shape[0]}, 列={data.shape[1]})")

        # ---- 5. 信号处理示例 ----
        eeg_channels = BoardShim.get_eeg_channels(board_id)
        sampling_rate = BoardShim.get_sampling_rate(board_id)

        for ch in eeg_channels[:3]:  # 只演示前 3 个通道
            channel_data = data[ch].copy()

            # 去趋势:去除信号中缓慢变化的“漂移”成分
            DataFilter.detrend(channel_data, DetrendOperations.LINEAR.value)

            # 带通滤波 1-50Hz
            DataFilter.perform_bandpass(
                channel_data,
                sampling_rate,
                1.0,    # 低截止频率
                50.0,   # 高截止频率
                4,      # 滤波器阶数
                FilterTypes.BUTTERWORTH.value,
                0.0     # ripple
            )

            # 计算频段功率
            psd = DataFilter.get_psd_welch(
                channel_data,
                256,           # FFT 大小
                128,           # 重叠
                sampling_rate,
                WindowOperations.HANNING
            )

            # 各频段功率
            delta = DataFilter.get_band_power(psd, 1.0, 4.0)
            theta = DataFilter.get_band_power(psd, 4.0, 8.0)
            alpha = DataFilter.get_band_power(psd, 8.0, 13.0)
            beta  = DataFilter.get_band_power(psd, 13.0, 30.0)

            print(f"通道 {ch}:")
            print(f"  δ={delta:.2f}  θ={theta:.2f}  α={alpha:.2f}  β={beta:.2f}")

        # ---- 6. 插入标记 ----
        board.insert_marker(1.0)
        print("\n标记已插入 ✓")

        # ---- 7. 停止 ----
        board.stop_stream()
        print("数据流已停止 ✓")

    finally:
        board.release_session()
        print("会话已释放 ✓")


if __name__ == "__main__":
    main()


采样率: 250 Hz
EEG 通道: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
EXG 通道: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16]
加速度通道: [17, 18, 19]
时间戳通道: 30
标记通道: 31
总行数: 32
加速度通道: [17, 18, 19]
会话已准备 ✓
数据流已启动，采集 5 秒...

数据形状: (32, 256)  (行=32, 列=256)
通道 1:
  δ=16.48  θ=20.98  α=0.00  β=0.00
通道 2:
  δ=0.04  θ=5.62  α=148.15  β=0.08
通道 3:
  δ=0.07  θ=0.24  α=7.53  β=337.86

标记已插入 ✓
数据流已停止 ✓
会话已释放 ✓
